# Build & track the ML pipeline with DVC: walkthrough

Companion to `notes/12_ml_pipelines_with_dvc.html` (video 02:46 – 03:09). Run it from the **project root** (`ch12_dvc_pipeline/`).

What happens, in the same order as the video:
1. run a component by hand → 2. `git init` + `dvc init` → 3. `dvc.yaml` with **one** stage → `dvc repro`
4. re-run (skipped) → 5. edit the code (re-runs) → 6. add the other stages one by one
7. `dvc dag`, `dvc metrics show` → 8. extras: `dvc status`, `dvc metrics diff`, a deleted output, what to commit.

> ⚠️ The first code cell **resets** this folder's generated state (`.git`, `.dvc`, `dvc.lock`, `data/`, `model.pkl`, `metrics.json`, `logs/`) so the notebook can be re-run from scratch. Your `src/`, `notebooks/` and `dvc.yaml` are kept.

In [1]:
import os, sys, shutil, json, pathlib, yaml, certifi

PROJECT = pathlib.Path.cwd()
assert (PROJECT / "src" / "data_ingestion.py").exists(), "open this notebook from the ch12_dvc_pipeline folder"

# make `dvc` and `python` resolve to this environment inside %%bash cells
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
os.environ["DVC_NO_ANALYTICS"] = "1"
os.environ["SSL_CERT_FILE"] = certifi.where()
# isolated git identity (doesn't touch ~/.gitconfig)
os.environ.update(GIT_AUTHOR_NAME="Learner", GIT_AUTHOR_EMAIL="learner@example.com",
                  GIT_COMMITTER_NAME="Learner", GIT_COMMITTER_EMAIL="learner@example.com",
                  GIT_CONFIG_GLOBAL=os.devnull, GIT_PAGER="cat")
for k in ("CLICOLOR", "CLICOLOR_FORCE", "LS_COLORS"):
    os.environ.pop(k, None)

# reset generated state only
for name in [".git", ".dvc", "data", "logs"]:
    shutil.rmtree(PROJECT / name, ignore_errors=True)
for name in ["dvc.lock", "model.pkl", "metrics.json", ".dvcignore", ".gitignore"]:
    (PROJECT / name).unlink(missing_ok=True)

# a normal Python .gitignore (DVC will append its own entries to it later)
(PROJECT / ".gitignore").write_text("\n".join(["__pycache__/", "*.py[cod]", "logs/", ".ipynb_checkpoints/", ""]))

FULL_DVC_YAML = (PROJECT / "dvc.yaml").read_text()        # the finished pipeline, restored at the end
STAGES = yaml.safe_load(FULL_DVC_YAML)["stages"]

def write_pipeline(n_stages: int) -> None:
    """Write dvc.yaml containing only the first n stages (to add them one by one, as in the video)."""
    if n_stages == len(STAGES):
        (PROJECT / "dvc.yaml").write_text(FULL_DVC_YAML)
    else:
        subset = dict(list(STAGES.items())[:n_stages])
        (PROJECT / "dvc.yaml").write_text(yaml.safe_dump({"stages": subset}, sort_keys=False))
    print((PROJECT / "dvc.yaml").read_text())

print("project:", PROJECT)
print("stages available:", list(STAGES))

project: /Users/hemanthreddy/Desktop/data_science/MLOps/projects/ch12_dvc_pipeline
stages available: ['data_ingestion', 'data_preprocessing', 'feature_engineering', 'model_building', 'model_evaluation']


## 1 · Each component runs on its own  (video 02:48)
Before DVC, the instructor tests each file with `python src/<file>.py`. Here's ingestion; then the output is deleted again so DVC can recreate it.

In [2]:
%%bash
python src/data_ingestion.py
ls -l data/raw
rm -rf data logs

2026-09-16 20:19:03,555 | data_ingestion | INFO | loaded 40000 rows from https://raw.githubuserconte

nt.com/entbappy/Branching-tutorial/refs/heads/master/tweet_emotions.csv


2026-09-16 20:19:03,557 | data_ingestion | INFO | kept 10374 rows (happiness=1, sadness=0)


2026-09-16 20:19:03,570 | data_ingestion | INFO | saved 8299 train / 2075 test rows to data/raw


total 1608
-rw-r--r--  1 hemanthreddy  staff  162098 Sep 16 20:19 test.csv
-rw-r--r--  1 hemanthredd

y  staff  655499 Sep 16 20:19 train.csv


## 2 · `git init` → `dvc init`  (video 02:57)
DVC runs on top of Git, so the folder must be a Git repository.

In [3]:
%%bash
git init -q -b main && echo "git initialised"
dvc init
echo "--- new files ---"
ls -a
echo "--- .dvc/ ---"
ls -a .dvc

git initialised


Initialized DVC repository.



You can now commit the changes to git.



What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share 

ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>


--- new files ---


.
..
.dvc
.dvcignore
.git
.gitignore
02_dvc_pipeline_walkthrough.ipynb
dvc.yaml
notebooks
requiremen

ts.txt
src


--- .dvc/ ---


.
..
.gitignore
config
tmp


## 3 · `dvc.yaml` with just the first stage → `dvc repro`  (video 02:55 – 02:59)
Every stage has `cmd` (what to run), `deps` (script + inputs) and `outs` (what it produces).

In [4]:
write_pipeline(1)

stages:
  data_ingestion:
    cmd: python src/data_ingestion.py
    deps:
    - src/data_ingestion.py
    outs:
    - data/raw



In [5]:
%%bash
dvc repro 2>&1

Running stage 'data_ingestion':


> python src/data_ingestion.py


2026-09-16 20:19:05,601 | data_ingestion | INFO | loaded 40000 rows from https://raw.githubuserconte

nt.com/entbappy/Branching-tutorial/refs/heads/master/tweet_emotions.csv


2026-09-16 20:19:05,604 | data_ingestion | INFO | kept 10374 rows (happiness=1, sadness=0)


...

Generating lock file 'dvc.lock'


Updating lock file 'dvc.lock'



To track the changes with git, run:

	git add dvc.lock data/.gitignore



To enable auto staging, run:

	dvc config core.autostage true


Use `dvc push` to send your updates to remote storage.


`dvc repro` wrote **`dvc.lock`**, a fingerprint (md5 hash + size) of every dependency and output.
This is how DVC "remembers" what it ran:

In [6]:
%%bash
cat dvc.lock

schema: '2.0'
stages:
  data_ingestion:
    cmd: python src/data_ingestion.py
    deps:
    - path: 

src/data_ingestion.py
      hash: md5
      md5: d9c536572a7b02f4ae15797bffb64359
      size: 1802
 

   outs:
    - path: data/raw
      hash: md5
      md5: 6d7c9717b531961233cd66af69381384.dir
      

size: 817597
      nfiles: 2


## 4 · Run it again: nothing to do  (video 03:00)

In [7]:
%%bash
dvc repro 2>&1

Stage 'data_ingestion' didn't change, skipping


Data and pipelines are up to date.


## 5 · Change the code: the stage re-runs  (video 03:01)
As in the video, switch the positive class from **happiness** to **neutral**.

In [8]:
src_file = PROJECT / "src" / "data_ingestion.py"
original_code = src_file.read_text()
src_file.write_text(original_code.replace('POSITIVE_CLASS = "happiness"', 'POSITIVE_CLASS = "neutral"'))
print([l for l in src_file.read_text().splitlines() if l.startswith("POSITIVE_CLASS")])

['POSITIVE_CLASS = "neutral"   # the demo later swaps this to "neutral" to show DVC re-running the stage']


In [9]:
%%bash
dvc status
dvc repro 2>&1

data_ingestion:
	changed deps:
		modified:           src/data_ingestion.py


Running stage 'data_ingestion':


> python src/data_ingestion.py


2026-09-16 20:19:08,227 | data_ingestion | INFO | loaded 40000 rows from https://raw.githubuserconte

nt.com/entbappy/Branching-tutorial/refs/heads/master/tweet_emotions.csv


...

2026-09-16 20:19:08,246 | data_ingestion | INFO | saved 11042 train / 2761 test rows to data/raw


Updating lock file 'dvc.lock'



To track the changes with git, run:

	git add dvc.lock



To enable auto staging, run:

	dvc config core.autostage true


Use `dvc push` to send your updates to remote storage.


In [10]:
import pandas as pd
print("rows in data/raw/train.csv:", len(pd.read_csv("data/raw/train.csv")),
      "(neutral + sadness, so more rows than happiness + sadness)")
src_file.write_text(original_code)   # change it back, like the instructor
print([l for l in src_file.read_text().splitlines() if l.startswith("POSITIVE_CLASS")])

rows in data/raw/train.csv: 11042 (neutral + sadness, so more rows than happiness + sadness)
['POSITIVE_CLASS = "happiness"   # the demo later swaps this to "neutral" to show DVC re-running the stage']


Changing the code **back** makes the stage "changed" again. Recent DVC versions have a **run-cache**, so if this exact combination was run before, DVC restores the old outputs instead of recomputing (look for *"cached"* / *"checking out"* in the output):

In [11]:
%%bash
dvc repro 2>&1

Stage 'data_ingestion' is cached - skipping run, checking out outputs


Updating lock file 'dvc.lock'



To track the changes with git, run:

	git add dvc.lock



To enable auto staging, run:

	dvc config core.autostage true


Use `dvc push` to send your updates to remote storage.


## 6 · Add the remaining stages one at a time  (video 03:03 – 03:06)
Each time, the earlier stages are **skipped** and only the new stage runs.

In [12]:
write_pipeline(2)   # + data_preprocessing

stages:
  data_ingestion:
    cmd: python src/data_ingestion.py
    deps:
    - src/data_ingestion.py
    outs:
    - data/raw
  data_preprocessing:
    cmd: python src/data_preprocessing.py
    deps:
    - data/raw
    - src/data_preprocessing.py
    outs:
    - data/processed



In [13]:
%%bash
dvc repro 2>&1

Stage 'data_ingestion' didn't change, skipping


Running stage 'data_preprocessing':


> python src/data_preprocessing.py


2026-09-16 20:19:11,885 | data_preprocessing | INFO | train: 8298 rows written


2026-09-16 20:19:11,921 | data_preprocessing | INFO | test: 2074 rows written


Updating lock file 'dvc.lock'



To track the changes with git, run:

	git add data/.gitignore dvc.lock



To enable auto staging, run:

	dvc config core.autostage true


Use `dvc push` to send your updates to remote storage.


In [14]:
write_pipeline(3)   # + feature_engineering

stages:
  data_ingestion:
    cmd: python src/data_ingestion.py
    deps:
    - src/data_ingestion.py
    outs:
    - data/raw
  data_preprocessing:
    cmd: python src/data_preprocessing.py
    deps:
    - data/raw
    - src/data_preprocessing.py
    outs:
    - data/processed
  feature_engineering:
    cmd: python src/feature_engineering.py
    deps:
    - data/processed
    - src/feature_engineering.py
    outs:
    - data/features



In [15]:
%%bash
dvc repro 2>&1

Stage 'data_ingestion' didn't change, skipping


Stage 'data_preprocessing' didn't change, skipping


Running stage 'feature_engineering':


> python src/feature_engineering.py


2026-09-16 20:19:14,215 | feature_engineering | INFO | train features: (8298, 1000)


2026-09-16 20:19:14,375 | feature_engineering | INFO | test features: (2074, 1000)


Updating lock file 'dvc.lock'



To track the changes with git, run:

	git add data/.gitignore dvc.lock



To enable auto staging, run:

	dvc config core.autostage true


Use `dvc push` to send your updates to remote storage.


In [16]:
write_pipeline(4)   # + model_building

stages:
  data_ingestion:
    cmd: python src/data_ingestion.py
    deps:
    - src/data_ingestion.py
    outs:
    - data/raw
  data_preprocessing:
    cmd: python src/data_preprocessing.py
    deps:
    - data/raw
    - src/data_preprocessing.py
    outs:
    - data/processed
  feature_engineering:
    cmd: python src/feature_engineering.py
    deps:
    - data/processed
    - src/feature_engineering.py
    outs:
    - data/features
  model_building:
    cmd: python src/model_building.py
    deps:
    - data/features
    - src/model_building.py
    outs:
    - model.pkl



In [17]:
%%bash
dvc repro 2>&1

Stage 'data_ingestion' didn't change, skipping


Stage 'data_preprocessing' didn't change, skipping


Stage 'feature_engineering' didn't change, skipping


Running stage 'model_building':


> python src/model_building.py


...

earning_rate': 0.1, 'eval_metric': 'logloss', 'random_state': 42}


Updating lock file 'dvc.lock'



To track the changes with git, run:

	git add dvc.lock .gitignore



To enable auto staging, run:

	dvc config core.autostage true


Use `dvc push` to send your updates to remote storage.


In [18]:
write_pipeline(5)   # + model_evaluation (the full file, with metrics:)

stages:
  data_ingestion:
    cmd: python src/data_ingestion.py
    deps:
      - src/data_ingestion.py
    outs:
      - data/raw

  data_preprocessing:
    cmd: python src/data_preprocessing.py
    deps:
      - data/raw
      - src/data_preprocessing.py
    outs:
      - data/processed

  feature_engineering:
    cmd: python src/feature_engineering.py
    deps:
      - data/processed
      - src/feature_engineering.py
    outs:
      - data/features

  model_building:
    cmd: python src/model_building.py
    deps:
      - data/features
      - src/model_building.py
    outs:
      - model.pkl

  model_evaluation:
    cmd: python src/model_evaluation.py
    deps:
      - model.pkl
      - src/model_evaluation.py
    metrics:
      - metrics.json



In [19]:
%%bash
dvc repro 2>&1

Stage 'data_ingestion' didn't change, skipping


Stage 'data_preprocessing' didn't change, skipping


Stage 'feature_engineering' didn't change, skipping


Stage 'model_building' didn't change, skipping


Running stage 'model_evaluation':


...

5, 'recall': 0.6252, 'auc': 0.8368}


Updating lock file 'dvc.lock'



To track the changes with git, run:

	git add .gitignore dvc.lock



To enable auto staging, run:

	dvc config core.autostage true


Use `dvc push` to send your updates to remote storage.


In [20]:
%%bash
echo "--- one more time: everything up to date ---"
dvc repro 2>&1

--- one more time: everything up to date ---


Stage 'data_ingestion' didn't change, skipping


Stage 'data_preprocessing' didn't change, skipping


Stage 'feature_engineering' didn't change, skipping


Stage 'model_building' didn't change, skipping


Stage 'model_evaluation' didn't change, skipping


Data and pipelines are up to date.


## 7 · `dvc dag` and `dvc metrics show`  (video 03:07)

In [21]:
%%bash
dvc dag

  +----------------+     
  | data_ingestion |     
  +----------------+     
            *         

   
            *            
            *            
+--------------------+   
| data_preprocessi

ng |   
+--------------------+   
            *            
            *            
            * 

           
+---------------------+  
| feature_engineering |  
+---------------------+  
          

  *            
            *            
            *            
  +----------------+     
  | mo

del_building |     
  +----------------+     
            *            
            *            
  

          *            
  +------------------+   
  | model_evaluation |   
  +------------------+  

In [22]:
%%bash
dvc metrics show
echo "--- metrics.json ---"
cat metrics.json

Path          accuracy    auc     precision    recall
metrics.json  0.7401      0.8368  0.7995      

 0.6252


--- metrics.json ---


{
    "accuracy": 0.7401,
    "precision": 0.7995,
    "recall": 0.6252,
    "auc": 0.8368
}

---
## 8 · Beyond the video

### 8.1 · What to commit
DVC has already added its outputs to `.gitignore` files, so large data never goes into Git. You commit the **pipeline definition + lock file**:

In [23]:
%%bash
echo "--- data/.gitignore ---"; cat data/.gitignore
echo "--- /.gitignore ---"; cat .gitignore
git add .
git status --short
git commit -q -m "pipeline: 5 DVC stages" && git log --oneline

--- data/.gitignore ---


/raw
/processed
/features


--- /.gitignore ---


__pycache__/
*.py[cod]
logs/
.ipynb_checkpoints/
/model.pkl
/metrics.json


A  .dvc/.gitignore
A  .dvc/config
A  .dvcignore
A  .gitignore
A  02_dvc_pipeline_walkthrough.ipynb
A

  data/.gitignore
A  dvc.lock
A  dvc.yaml
A  notebooks/01_experiment_twitter_emotion.ipynb
A  requir

ements.txt
A  src/data_ingestion.py
A  src/data_preprocessing.py
A  src/feature_engineering.py
A  sr

c/model_building.py
A  src/model_evaluation.py
A  src/utils.py


741e26a pipeline: 5 DVC stages


### 8.2 · Change only the model → only 2 stages re-run, then compare metrics

In [24]:
mb = PROJECT / "src" / "model_building.py"
mb_original = mb.read_text()
mb.write_text(mb_original.replace('"n_estimators": 100', '"n_estimators": 300'))
print([l for l in mb.read_text().splitlines() if l.startswith("PARAMS")])

['PARAMS = {"n_estimators": 300, "learning_rate": 0.1, "eval_metric": "logloss", "random_state": 42}']


In [25]:
%%bash
dvc status
dvc repro 2>&1
echo "--- metrics vs the last commit ---"
dvc metrics diff

model_building:
	changed deps:
		modified:           src/model_building.py


Stage 'data_ingestion' didn't change, skipping


Stage 'data_preprocessing' didn't change, skipping


Stage 'feature_engineering' didn't change, skipping


Running stage 'model_building':


...

Use `dvc push` to send your updates to remote storage.


--- metrics vs the last commit ---


Path          Metric     HEAD    workspace    Change
metrics.json  accuracy   0.7401  0.7748       0

.0347
metrics.json  auc        0.8368  0.8601       0.0233
metrics.json  precision  0.7995  0.7957  

     -0.0038
metrics.json  recall     0.6252  0.7258       0.1006


In [26]:
_ = mb.write_text(mb_original)   # restore the committed version

In [27]:
%%bash
dvc repro 2>&1 | tail -4
dvc metrics diff && echo "(no difference: back to the committed model)"

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to

 remote storage.


(no difference: back to the committed model)


### 8.3 · Accidentally deleted output → `dvc status` notices, `dvc repro` fixes it

In [28]:
%%bash
rm model.pkl
dvc status
dvc repro 2>&1 | tail -6
ls -l model.pkl

model_building:
	changed outs:
		deleted:            model.pkl
model_evaluation:


	changed deps:
		deleted:            model.pkl


Stage 'data_preprocessing' didn't change, skipping
Stage 'feature_engineering' didn't change, skippi

ng
Stage 'model_building' is cached - skipping run, checking out outputs

Stage 'model_evaluation' d

idn't change, skipping
Use `dvc push` to send your updates to remote storage.


-rw-r--r--  1 hemanthreddy  staff  176818 Sep 16 20:19 model.pkl


### 8.4 · The URL trap
`data_ingestion` only depends on its **script**. If new rows were added to the CSV **at the URL**, `dvc status` would still report *up to date* and the stage would be skipped.
Fixes: `dvc import-url <url> data/source.csv` + `dvc update`, or `always_changed: true` on that stage, or a data-version value in `params.yaml`.

In [29]:
%%bash
dvc status

Data and pipelines are up to date.


## Summary

| Step | Command | What we saw |
|---|---|---|
| set up | `git init`, `dvc init` | `.dvc/`, `.dvcignore` created |
| run pipeline | `dvc repro` | only stages whose deps/outs changed run; `dvc.lock` updated |
| nothing changed | `dvc repro` | *didn't change, skipping* |
| code changed | `dvc repro` | that stage + everything downstream re-runs |
| inspect | `dvc dag`, `dvc metrics show`, `dvc status` | graph, metrics table, what's out of date |
| compare | `dvc metrics diff` | metric change vs last commit |
| commit | `git add dvc.yaml dvc.lock .gitignore …` | data stays out of Git |

Re-run this notebook from the top at any time: the first cell resets the generated state.